In [ ]:
#importing libraries
from src.MFE import *
from src.MyUlamGalerkin import *
from src.LeichtNewman import *
from src.MST import *
from src.helper import *
from src.visulization import *
from src.SnapCluster import SnapCluster
from src.UlamGalerkin import UlamGalerkin
from src.Dijkstras import *

import time
import numpy as np
import math 
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation
from matplotlib.animation import FFMpegWriter
from pathlib import Path
from sklearn.cluster import KMeans
import networkx as nx
from collections import Counter
import matplotlib.ticker as mticker
import sys
from joblib import Parallel, delayed
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
np.set_printoptions(suppress=True) # to avoid scientific notation in numpy printing


In [2]:
#Plot Settings


plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['font.family']='serif'

In [ ]:
#Start the MFE sequence generation


dt,N,istart =0.05,510000,10000      # time-step and number of trajectory points
Re=800
Lx,Lz=1.75*np.pi,1.2*np.pi
D,E,a = MFE_Sequence(dt,N,istart,Re,Lx,Lz)   # generate data sequence 
#t=(np.arange(len(D))+1)*dt
DE = np.vstack((D,E)).T

caseName=""
Case=create_case_folder(dt,N-istart,Re,Lx,Lz)  
Case = os.path.join(Case, caseName)  

figo, axo = plt.subplots()

t=N-istart-1
axo.plot(DE[:, 1], DE[:, 0])
axo.scatter(DE[t, 1], DE[t, 0],c="red",zorder=10)
axo.set_title(f'MFE Sim, dt={dt}, N={N-istart}, Re={Re}')
path = Path(os.path.join(Case, f"DE_N={N-istart}_dt={dt}_Re={Re}.png"))
axo.set_xlabel("Energy")
axo.set_ylabel("Dissipation")
#axo.set_facecolor("#EAEBF4")
figo.savefig(path)  

np.savetxt(
    os.path.join(Case, "DE.csv"),
    DE,
    delimiter=",",
    header="D,E",
    comments="",
)
print(F"Time: {dt*(N-istart)}s")


In [ ]:
plot_disctrized_phase_space(DE,50)

In [ ]:
###Ulam-Galerkin###
Disctrize_box_size=50 #Chose the box size for Ulam-Galerkin
A= SnapCluster(DE,Disctrize_box_size)

clust_count=A.cluster_counts()
#print(clust_count)
print(f"maximum number of snapshots inside a cluster: {max(clust_count)}")
#Plot the transition matrix
'''
###LeichtNewman
A.repeated_leicht_newman_algorithm(k=5)
Q = compute_Modularity(A.CPT)
print(f"Modularity Q: {Q}")
#plot_dijkstra_shortest_path(CP, DE, start=164)
#print(f"Q={SnapCluster().compQ(CPT)}")
nclust=max(A.CPT[-1])+1
colors, ci = cluster_colors(A.CPT)
plt.scatter(DE[:, 1], DE[:, 0], s=5, c=colors)
plt.xlabel("Energy")
plt.ylabel("Dissipation")
plt.title(f"LeichtNewman, Clusters: {nclust}")
#plt.axvline(x=critical_points[1], color='red', linestyle='--', label='Critical Points')
#plt.axhline(y=critical_points[0], color='red', linestyle='--', label='Critical Points')
plt.savefig(os.path.join(Case, f"LeichtNewman_reclustering_nclust={nclust}.png"))
plt.show()
'''

###MST_kruskal

Cluster_size=6
#CP=A.CPT[-1]
#print(f"Number of old clusters: {np.unique(CP).shape[0]}")
CPT=A.repeated_mst_algorithm(Cluster_size,method="kruskal",cluster_type="boltzman")
Q = compute_Modularity(A.CPT)
print(f"Modularity Q: {Q}")

nclust=max(A.CPT[-1])+1
colors, ci = cluster_colors(A.CPT)
colors = np.array(colors)

figo, ax = plt.subplots()
ax.scatter(DE[:, 1], DE[:, 0], s=5, c=colors)
ax.set_xlabel("Energy")
ax.set_ylabel("Dissipation")
ax.set_title(f"MST, Clusters: {nclust}")
figo.savefig(os.path.join(Case, f"MST_reclustering_nclust={nclust}.png"))
#plot_box_transition_matrix(A.CPT)
#A.save(Case)

In [ ]:
kc=-10
colors, ci = cluster_colors(A.CPT,k=kc)
nclust=max(A.CPT[kc])+1
colors = np.array(colors)

figo, ax = plt.subplots()
ax.scatter(DE[:, 1], DE[:, 0], s=5, c=colors)
ax.set_xlabel("Energy")
ax.set_ylabel("Dissipation")
ax.set_title(f"MST, Clusters: {nclust}")

In [ ]:

P0M,bx_clusters=box_cluster_transition_matrix(CPT)
bx_clusters_count=list(bx_clusters.values())
unique_vals, cluster_size = np.unique(bx_clusters_count, return_counts=True)


fig_trans = plt.figure(figsize=(10, 8),dpi=500)
ax_trans = plt.gca()
im_trans = ax_trans.imshow(P0M, cmap='Blues',norm=mcolors.LogNorm(vmin=1e-6, vmax=0.5))
ax_trans.set_xlabel('From Box')
ax_trans.set_ylabel('To Box')
ax_trans.set_ylim(P0M.shape[0],-1)
ax_trans.set_xlim(-1, P0M.shape[1])

ax_trans.set_title('Cluster Transition Probability Matrix')
line=-0.5

for i in range(len(cluster_size)):
        plt.vlines(x=line, ymin=line, ymax=cluster_size[i]+line, colors='red',linestyles='--',linewidth=0.5)
        plt.hlines(y=line, xmin=line, xmax=cluster_size[i]+line, colors='red',linestyles='--',linewidth=0.5)
        plt.vlines(x=cluster_size[i]+line, ymin=line, ymax=cluster_size[i]+line, colors='red',linestyles='--',linewidth=0.5)
        plt.hlines(y=cluster_size[i]+line, xmin=line, xmax=cluster_size[i]+line, colors='red',linestyles='--',linewidth=0.5)
        line+=cluster_size[i]


#ax_trans.yaxis.set_major_locator(MultipleLocator(1))
#ax_trans.xaxis.set_major_locator(MultipleLocator(1))
plt.colorbar(im_trans, ax=ax_trans, label='Probability')
plt.show()
    

In [ ]:
CC=cluster_centroids(A.CPT,DE)
cluster_center_points =[]
for i, center in enumerate(CC):
    distances = np.linalg.norm(DE - center, axis=1)
    closest_index = np.argmin(distances)
    cluster_center_points.append(closest_index)
cluster_center_points = np.array(cluster_center_points)
print(f"Cluster centers points: {cluster_center_points}")

In [ ]:
plot_markov_chain(A.CPT)
ex_points,critical_points= extreme_points(DE,2) 

plot_cluster_centers(Case,A.CPT,DE)
print(f"extreme clusters {np.unique(A.CPT[-1][ex_points])}")
print(critical_points)
#Cluster Centeres modes based on average of snapshots


laminarcluster=np.argmax(a[cluster_center_points,0])
cluster_count=A.cluster_counts()
plot_cluster_transition_matrix(A.CPT,method="CPT")
print(f"Cluster Time Percentage: {cluster_count/(N-istart)*100}")



In [ ]:
CP=A.CPT[-1]
Extreme_clusters= extreme_clusters(A.CPT, DE)
#extreme_clusters=np.append(extreme_clusters,1)
#extreme_clusters=np.append(extreme_clusters,9)
P=cluster_transition_matrix(A.CPT)
np.fill_diagonal(P,0)
P=P/P.sum(axis=0,keepdims=True)
non_extreme_clusters = [i for i in range(P.shape[0]) if i not in Extreme_clusters]
#extreme_clusters=np.add(extreme_clusters,18)

#####Print here######
print(f"extreme clusters {Extreme_clusters}") #####Print here######


Pej=np.zeros((P.shape[0]))
for j in range(P.shape[0]):
    for e in Extreme_clusters:
        Pej[j] += P[e,j]
    
Pex=[]
for i,p in enumerate(Pej):
    if np.isin(i, Extreme_clusters):
        continue
    else:
        Pex.append(p)
plt.Figure(figsize=(8, 6))
plt.bar(range(len(Pex)), Pex, tick_label=non_extreme_clusters, color='red',width=0.6,zorder=2)

plt.xlabel('Cluster')
plt.ylabel('Transition Probability to Ext Clusters')
plt.ylim(0, 1.05)
plt.title('Non-Extreme to Extreme Clusters')


    
plt.grid(axis='y')


In [ ]:
P=cluster_transition_matrix(A.CPT)
np.fill_diagonal(P,0)
P=P/P.sum(axis=0, keepdims=True)
plot_cluster_transition_matrix(P,"CTM")

In [ ]:
#dijkstra_shortest_path=dijkstra_algorithems(CP, start=2)
#print(dijkstra_shortest_path)
start =15
start0=start
path=[]
path.append(start)
#edges=dijkstra_algorithems(CP, start)
#print(edges)
for i in range(300):
    if i%100==0:
        print(f"it = {i}")
    filtered=[]
    edges=dijkstra_algorithems(CP, start)
    #print(edges)
    for e in edges:
        if e[2] != 0 and e[2] !=np.inf and (e[1]==start or e[0]==start):
            filtered.append(e)
    #print(filtered)
    #best_edge = min(filtered, key=lambda x: x[2])
    weights=[e[2] for e in filtered]
    #print(weights)
    weights=np.exp(-np.array(weights))
    sumW= sum(weights)
    #print(sumW)
    if sumW==0:
        break
    weights=weights/sumW
    #print(weights)
    #invweights=[]
    #invweights = 1 / np.array(weights)

    proabilities=weights#invweights/sum(invweights)
    #print(proabilities)

    choice = np.random.choice(len(proabilities), p=proabilities)
    best_edge = filtered[choice]
    #print(best_edge)
    if best_edge[1] == start:
        start = best_edge[0]
    else:
        start = best_edge[1]
    path.append(start)
    if start ==-1:
        path.pop() 
        break
print(path)
for i in path:
    plt.scatter(
        DE[CP == i, 1],
        DE[CP == i, 0],
        s=5,
        color=ci[i],
    )
#print(cluster_transition_matrix(CP))
plt.xlabel("Energy")
plt.ylabel("Dissipation")
plt.ylim(DE[:, 0].min() - 0.1, DE[:, 0].max() + 0.1)
plt.xlim(DE[:, 1].min() - 0.1, DE[:, 1].max() + 0.1)
plt.title(f"Dijkstra's shortest path from cluster {start0}")
plt.show()


In [ ]:


def print_mode_coefficients(a,Data,CP):
    #print("a vector at points closet to cluster centers:")
    for i in range(len(cluster_center_points)):
        print(f"Point {i}: {DE[cluster_center_points[i]]}")
        print(f"{a[cluster_center_points[i]]}")
print_mode_coefficients(a,DE,CP)
#plotModeCoefficients(Case,a, N-istart, dt,"black")

In [ ]:
plotClusterMembership(Case,A.CPT, len(DE), dt)

In [ ]:
#plotDissipationAndEnergy(Case,DE,colors,len(DE),dt)
colors,ssss=cluster_colors(A.CPT)
plotDissipationAndEnergy(Case,DE,int(N-istart),dt,colors)

In [ ]:
#Distance matrix between clusters Euclidian
plot_cluster_distance_matrix(Case,A.CPT, DE)

In [ ]:
it = 10000000
CTM=cluster_transition_matrix(A.CPT)
CTMP=CTM_power(CTM,it)
fig_pow = plt.figure(figsize=(10, 8))
ax_pow = plt.gca()
im_pow = ax_pow.imshow(CTMP, cmap='YlOrRd',norm=mcolors.LogNorm(vmin=1e-3, vmax=CTMP.max()))
ax_pow.set_xlabel('From Cluster')
ax_pow.set_ylabel('To Cluster')
ax_pow.set_title(f'P^{it} (Transition Probabilities After {it} Steps)')
ax_pow.yaxis.set_major_locator(MultipleLocator(1))
ax_pow.xaxis.set_major_locator(MultipleLocator(1))
plt.colorbar(im_pow, ax=ax_pow, label='Probability')
plt.show()

In [ ]:
#Fixed Point Probability Vector
q=A.fixed_point_vector()
nclust=max(CP)+1
fig6 = plt.figure(figsize=(10,5))
ax6 = fig6.add_subplot(121)
ax6.bar(np.arange(nclust), q, color='skyblue', edgecolor='black')
ax6.set_xlabel('Cluster ID')
ax6.set_ylabel('$q_k$')
ax6.set_title(f"Fixed Point Probability Vector")

#Probability Vector after L iterations
L=100000
p0=np.zeros(nclust)
p0[0]=1 #Initial Condition of the probability vector
p0=p0/np.sum(p0)
p=CTM_power(CTM,L) @ p0
ax6 = fig6.add_subplot(122)
ax6.bar(np.arange(nclust), p, color='skyblue', edgecolor='black')
ax6.set_xlabel('Cluster ID')
ax6.set_ylabel('$p_k$')
ax6.set_title(f"Probability Vector after {L} iterations")
plt.show()

In [ ]:
#Dynamics Of probability Vector
L = 1000000
p0 = np.zeros(nclust)
p0[[0,3]] = [0.1,0]  # Initial condition
p0=p0/np.sum(p0)

it_vals = []  # To store iterations
p1_vals = []  # To store p[0] values

p9_vals = []  # To store p[0] values
p_vals = []  # To store the entire probability vector at each iteration
p=p0.copy()
CTM=cluster_transition_matrix(A.CPT)
for i in range(L):
    p = CTM @ p
    it_vals.append(i)   # Iteration number (1-based)
    p_vals.append(p)  # Probability of first cluster
    p1_vals.append(p[0])  # Probability of first cluster
    p9_vals.append(p[1])  # Probability of first cluster
p_vals = np.array(p_vals).T  # Convert list of arrays to a 2D array (iterations x clusters)
fig7 = plt.figure()
ax7 = fig7.add_subplot(111)
for i in range(len(p_vals)):
    pi = p_vals[i]
    ax7.plot(it_vals, pi, label=f'p_{i}', linestyle='-')

ax7.set_xlabel("Iteration")
ax7.set_ylabel("Probability")
ax7.set_title("Dynamics of Probability Vector")
ax7.legend()
plt.show()

In [ ]:
L=20000
CTM=cluster_transition_matrix(A.CPT)
it = []
eigenvalue_2 = []
for i in range(L):
    eigenvalues_full, eigenvectors = np.linalg.eig(CTM_power(CTM,i))
    eigenvalues_full_sorted = sorted(eigenvalues_full, key=abs, reverse=True)
    it.append(i+1)   # Iteration number (1-based)
    eigenvalue_2.append(abs(eigenvalues_full_sorted[1]))  # Second largest eigenvalue
fig8 = plt.figure()
ax8 = fig8.add_subplot(111)
ax8.plot(it, eigenvalue_2, color='b', linestyle='-')
ax8.set_xscale('log')
ax8.set_xlabel("Iteration")
ax8.set_ylabel("eigenvalue")
ax8.set_title("Second Largest Eigenvalue of CTM^L")
plt.show()

In [ ]:
#flow norm variance R(p)^2
L=20000
p0 = np.zeros(nclust)
p0[:] = 1  # Initial condition
p0=p0/np.sum(p0)
it = []
R_2=[]
p = p0.copy()
D_sq = D**2
CTM=cluster_transition_matrix(A.CPT)
for i in range(L):
    p = CTM @ p
    p=p/np.sum(p)
    R_2_sum = np.sum(np.outer(p, p) * D_sq)
    R_2.append(R_2_sum)
    it.append(i+1)   # Iteration number 
fig9=plt.figure()
ax9 = fig9.add_subplot(111)
ax9.plot(it, R_2, color='b', linestyle='-')
ax9.set_xscale('log')
ax9.set_xlabel("Iteration")
ax9.set_ylabel("R^2")
ax9.set_title("R^2 vs Iteration")    

In [ ]:
L=200000
Q=q[:,None]@np.ones((1, nclust))
H=[]
it=[]
CTM=cluster_transition_matrix(A.CPT)
for i in range(L):
    CTMP=CTM_power(CTM,i)
    mask = (CTMP > 0) & (Q > 0)
    H_sum= -np.sum(CTMP[mask] * np.log(CTMP[mask] / Q[mask]))
    H.append(H_sum)
    it.append(i+1)   # Iteration number 

fig10=plt.figure()
ax10 = fig10.add_subplot(111)
ax10.plot(it, H, color='b', linestyle='-')
ax10.set_xscale('log')
ax10.set_xlabel("Iteration")
ax10.set_ylabel("H")
ax10.set_title("H vs Iteration")  

In [ ]:
#%%
sim_time=np.arange(0,int((N-istart)),20)
fig8, axis = plt.subplots()
axis.set_xlim(min(E), max(E))
axis.set_ylim(min(D), max(D))
axis.set_xlabel("Energy")#,color="white")
axis.set_ylabel("Dissipation")#,color="white")
axis.tick_params(axis="both")#,color='white')
'''
fig8.set_facecolor("black")
axis.set_facecolor("black")
for spine in axis.spines.values():
    spine.set_color('white')
for label in axis.get_xticklabels():
    label.set_color('white')
for label in axis.get_yticklabels():
    label.set_color('white')
'''
# Trajectory line in blue
trajectory_line = axis.scatter([], [], c=[], s=1)
# Current point in red
current_point, = axis.plot([], [], 'ro', markersize=5)
t0 = time.time()
colors,c=cluster_colors(A.CPT,k=-9)

def update(frame):
    h = min(frame, len(E))  # avoid exceeding array length
    #x = E[:h]  # must be a sequence
    #y = D[:h]  # must be a sequence
    trajectory_line.set_offsets(np.c_[E[:h], D[:h]],)
    trajectory_line.set_color(colors[:h])
    # Highlight the current point
    current_point.set_data([E[h-1]], [D[h-1]])
    axis.set_title(f"Time: {frame*dt}s")#,color="white")
    return trajectory_line, current_point
t1 = time.time()
animation = FuncAnimation(fig=fig8, func=update, frames=sim_time, interval=50,blit=True)
# Save the animation
writer = FFMpegWriter(fps=10)
animation.save("Videos/test_o.mp4", writer=writer)
t2 = time.time()
plt.show()
print(f"time taken to animate ={t1-t0}")
print(f"time taken to save as MP4 ={t2-t1}")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter

fig, ax = plt.subplots()

ax.set_xlim(DE[:, 1].min(), DE[:, 1].max())
ax.set_ylim(DE[:, 0].min(), DE[:, 0].max())

# --------------------------------------------------
# PRECOMPUTE cluster indices (FAST instead of masks each frame)
# --------------------------------------------------
clusters = np.unique(CP)
cluster_idx = {c: np.where(CP == c)[0] for c in clusters}

# map each cluster → points
cluster_xy = {
    c: DE[idx][:, [1, 0]] for c, idx in cluster_idx.items()
}

# --------------------------------------------------
# PRECOMPUTE cumulative frames (FAST union via indices)
# --------------------------------------------------
visited = set()
frames_points = []
frames_colors = []

for c in path:
    visited.add(c)

    idx_all = np.concatenate([cluster_idx[v] for v in visited])

    pts = DE[idx_all][:, [1, 0]]
    frames_points.append(pts)

# store cluster per point per frame (for coloring)
for c in path:
    visited.add(c)

    idx_all = np.concatenate([cluster_idx[v] for v in visited])

    frame_colors = np.array(colors)[idx_all]

    # highlight current cluster in red
    current_idx = cluster_idx[c]
    # mark which indices are in current frame selection
    mask = np.isin(idx_all, current_idx)

    frame_colors[mask] = (0.839, 0.152, 0.156, 1.0)

    frames_colors.append(frame_colors)

# --------------------------------------------------
# SCATTER INIT
# --------------------------------------------------
scat = ax.scatter([], [], s=5)

# --------------------------------------------------
# UPDATE (VERY FAST)
# --------------------------------------------------
def update(i):
    scat.set_offsets(frames_points[i])
    scat.set_color(frames_colors[i])
    ax.set_title(f"Step {i} | Cluster {path[i]}")
    return scat,

# --------------------------------------------------
# ANIMATION
# --------------------------------------------------
ani = FuncAnimation(
    fig,
    update,
    frames=len(path),
    interval=300,
    blit=False
)

# --------------------------------------------------
# SAVE (FAST)
# --------------------------------------------------
writer = FFMpegWriter(fps=2)
ani.save("DE.mp4", writer=writer)

plt.show()

In [ ]:
#K-means Clustering 
if __name__=='__main__':
    
    dt,N,istart = 0.05,510000,10000       # time-step and number of trajectory points
    Re=800
    D,E,a = MFE_Sequence(dt,N,istart,Re,2*np.pi,2*np.pi)   # generate data sequence 
    nclust = 10                # number of clusters ('boxes')  
    
    plt.figure(1)             # visualization 
    plt.ion()
    plt.clf()
    plt.plot(E,D)
    plt.gca().set_aspect(0.1)
    plt.show()
    plt.pause(0.001)

    DE = np.vstack((D,E)).T
    print('clustering in phase space') 

    km = KMeans(n_clusters=nclust, init='k-means++').fit(DE)  # 'discretize' into clusters 
    C  = km.labels_                     # cluster membership
    print(C.shape) 
    CC = km.cluster_centers_            # cluster centroids 
    c  = cm.tab10(np.linspace(0, 1, nclust))  # color coding
    cluster_counts = np.bincount(C, minlength=nclust)
    print(f"The minimum number of snapshots per cluster = {min(cluster_counts)}")
    colors = [c[label % len(c)] for label in C]
    for i in range(len(CC)):
        ii = np.where(C==i)[0]
        DD,EE = D[ii],E[ii]
    plt.scatter(E,D,s=5,c=colors)
    plt.show()
    print(CC)
